## Tahap 0 — Persiapan Lingkungan

**Package:** `os`, `sys`, `json`, `platform`, `random`, `warnings`, `time`, `math`, `subprocess`, `importlib`, `pathlib`, `datetime` (bawaan Python); `numpy`, `pandas`, `torch`, `transformers`, `scikit-learn`, `scipy`, `matplotlib`, `openpyxl`, `pyarrow`; `google.colab.drive`.

Jalankan sel ini **paling awal** setiap kali *runtime* dimulai atau di-*restart*. Sel ini membuat `PATH`, `CFG`, `TARGET`, dan fungsi bantu yang dipakai semua tahap.

In [ ]:
# ══ Tahap 0 — Persiapan Lingkungan ══
# Package : os, sys, json, platform, random, warnings, time, math, subprocess,
#           importlib, pathlib, datetime (bawaan Python); numpy, pandas, torch, transformers,
#           scikit-learn, scipy, matplotlib, openpyxl, pyarrow; google.colab.drive
import os, sys, json, platform, random, warnings, time, math, subprocess
import importlib.util
from importlib.metadata import version as _versi_pkg
from pathlib import Path
from datetime import datetime

t0 = time.perf_counter()

# 1) Determinisme dan senyap: diatur SEBELUM torch/transformers di-import
_cublas_lama = os.environ.get("CUBLAS_WORKSPACE_CONFIG")
_cuda_sudah_aktif = "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized()
os.environ.update({
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",        # determinisme cuBLAS (Bagian 3.4)
    "TOKENIZERS_PARALLELISM": "false",           # senyap: peringatan paralelisme tokenizer
    "TRANSFORMERS_VERBOSITY": "error",           # senyap: log transformers
    "TRANSFORMERS_NO_ADVISORY_WARNINGS": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",         # senyap: progress bar unduhan Hugging Face Hub
    "HF_DATASETS_DISABLE_PROGRESS_BARS": "1",    # senyap: progress bar datasets
    "TQDM_DISABLE": "1",                         # senyap: semua progress bar tqdm
    "USE_TF": "0", "TF_CPP_MIN_LOG_LEVEL": "3",  # jangan muat TensorFlow beserta log-nya
})
warnings.filterwarnings("ignore")

# 2) Pasang hanya package yang belum ada
PAKET = {"numpy": "numpy", "pandas": "pandas", "torch": "torch", "transformers": "transformers",
         "sklearn": "scikit-learn", "scipy": "scipy", "matplotlib": "matplotlib",
         "openpyxl": "openpyxl", "pyarrow": "pyarrow"}          # nama modul -> nama pip
_kurang = [pip for mod, pip in PAKET.items() if importlib.util.find_spec(mod) is None]
if _kurang:
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_kurang],
                        capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError("pip install gagal:\n" + _r.stderr[-2000:])

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, transformers
transformers.logging.set_verbosity_error()       # sembunyikan "Some weights ... newly initialized"
transformers.logging.disable_progress_bar()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)         # Keputusan no. 10 (Bagian 9)
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "savefig.bbox": "tight"})

# 3) Google Drive dan struktur folder (Bagian 3.1)
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    try:
        drive.mount("/content/drive")
    except ValueError as e:
        raise RuntimeError("Mount Google Drive gagal. Pilih akun yang menyimpan folder skripsi, "
                           "centang SEMUA izin, lalu jalankan ulang sel ini.") from e
AKAR = Path("/content/drive/MyDrive/skripsi_wondr_v2")   # folder baru; skripsi_wondr_final tidak disentuh
if not AKAR.parent.is_dir():
    raise RuntimeError("Google Drive belum ter-mount. Jalankan ulang sel ini.")
PATH = {"akar": AKAR, "data": AKAR / "data", "lib": AKAR / "lib", "hasil": AKAR / "hasil",
        "tabel": AKAR / "bab4" / "tabel", "gambar": AKAR / "bab4" / "gambar", "log": AKAR / "log"}
for _p in PATH.values():
    _p.mkdir(parents=True, exist_ok=True)
PATH["xlsx"] = PATH["data"] / "data_wonder_bybni_3.xlsx"
(PATH["lib"] / "__init__.py").touch()            # agar "from lib.wondr_teks import ..." pasti jalan
os.chdir(AKAR)                                    # agar %%writefile cukup memakai lib/...
if str(AKAR) not in sys.path:
    sys.path.insert(0, str(AKAR))

# 4) Konfigurasi tetap (Bagian 4 dan Bagian 9) -- jangan diubah
CFG = {
    "checkpoint": "indobenchmark/indobert-base-p1",   # Rujukan: Wilie dkk. (2020)
    "ambang_rating": 4,                               # Keputusan: top-2-box, rating >= 4 -> 1 (Morgan & Rego, 2006)
    "proporsi": {"latih": 0.70, "validasi": 0.15, "uji": 0.15},   # Keputusan
    "seed_bagi": 42,                                  # Keputusan
    "kelompok_panjang": ["1-2", "3-10", ">10"],       # Keputusan (bukti Tahap 2)
    "q": 100, "b": 32,                                # Keputusan (Jalur 2): L_max = b*ceil(P_q/b), data latih saja
    "B": 16,                                          # Rujukan: Devlin dkk. (2019)
    "E": 4,                                           # Rujukan: Sun dkk. (2019)
    "patience": 2,                                    # Keputusan
    "clip_norm": 1.0,                                 # Keputusan
    "adam_betas": (0.9, 0.999), "adam_eps": 1e-8,     # Bawaan PyTorch
    "tanpa_wd": ("bias", "LayerNorm.weight"),         # Keputusan no. 1: mengikuti BERT asli
    "default": {"lr": 2e-5, "wr": 0.1, "wd": 0.01},   # Rujukan: Devlin dkk. (2019); Sun dkk. (2019)
    "ruang": {"lr": (1e-5, 5e-5), "wr": (0.0, 0.2), "wd": (0.0, 0.3)},   # Rujukan: Liu & Wang (2021); Bischl dkk. (2023)
    # dekode f in [0,1]^3: lr = 1e-5 * 5**f1 (skala log), wr = 0.2*f2, wd = 0.3*f3   (Hitungan)
    "seed_fitness": (42, 43),                         # Keputusan: fitness = rata-rata 2 seed
    "n_kandidat_rs": 50, "seed_rs": 2024,             # Keputusan
    "pso": {"s": 10, "G_max": 4,                      # Keputusan: 10 x (1 + 4) = 50 evaluasi
            "omega": 0.729, "phi_p": 1.49445, "phi_g": 1.49445,   # Rujukan: Eberhart & Shi (2000); Clerc & Kennedy (2002)
            "V_max": 1.0,                             # Rujukan: Eberhart & Shi (2000)
            "seed": 7},                               # Keputusan
    "M_ragam": 100, "J": 2,                           # Hitungan: 50 RS + 50 PSO; 2 seed per konfigurasi
    "seed_akhir": (42, 43, 44),                       # Keputusan: 3 skema x 3 seed
    "R": 10_000, "seed_bootstrap": 42,                # Rujukan: Efron & Tibshirani (1993)
    "alpha": 0.05, "m_uji": 3,                        # Rujukan: Dror dkk. (2017; 2018); alpha' = 0,05/3
    "daya": 0.80, "delta_target": 0.005,              # Rujukan: Bloom (1995)
    "kembaran_dekat": (339, 595, 2854, 3946, 4128),   # Data (temuan Tahap 7)
}

# 5) Angka target (Bagian 5 dan Bagian 9); Tahap 1-8 sudah diverifikasi ulang 56/56
TARGET = {
    1: {"baris": 7204, "kolom": ["nama", "rating", "ulasan", "tanggal", "likes", "balasan_dev", "versi"],
        "tanggal": ("2026-07-19", "2026-09-11"), "versi": ["1.6.0"],
        "per_bulan": {"2026-07": 1747, "2026-08": 3899, "2026-09": 1558}},
    2: {"baris": 7204, "rating": (1, 5), "kosong": {"balasan_dev": 4834},
        "dibalas_pct": {"1-4": 95.0, "5": 1.0}, "likes_nol_pct": 95.1, "spearman": -0.2483,
        "r3_panjang": (237, 366), "median_kata": [15, 15, 11, 3, 2], "emoji_simbol": 404,
        "huruf_ulang": 227, "huruf_ulang_pct": 3.2, "baca_ulang": 634, "duplikat_pct": 34.8,
        "label_ganda": (33, 1968), "nama_unik": 7153},
    3: {"baris": 7204, "kolom": ["review_id", "ulasan", "rating", "tanggal", "versi"]},
    4: {"per_rating": {1: 1305, 2: 283, 3: 366, 4: 492, 5: 4758}, "positif": 5250,
        "non_positif": 1954, "positif_pct": 72.88, "non_positif_pct": 27.12},
    5: {"berubah": [55, 404, 1518, 227, 634, 403], "kosong": 63, "kosong_rating": {5: 57, 4: 6},
        "hasil": 7141},
    6: {"kunci_kosong": 0, "kunci": 4372, "kunci_seri": 8, "ulasan_seri": 16, "dokumen": 4364,
        "dibuang": 2777, "rincian": (2688, 73, 16), "positif_pct": (72.6, 57.3)},
    7: {"grup": 4293, "latih": (3051, 1303, 1748), "validasi": (657, 282, 375),
        "uji": (656, 280, 376), "bocor": 0, "kembaran_dekat_di_uji": 5},
    8: {"non_positif_pct": {"latih": 42.7, "validasi": 42.9, "uji": 42.7},
        "panjang_pct": {"latih": (14.0, 47.1, 38.9), "validasi": (14.3, 46.9, 38.8),
                        "uji": (14.0, 47.1, 38.9)},
        "p_label": 0.9945, "p_panjang": 0.9998,
        "acuan_validasi": (0.5708, 0.3634), "acuan_uji": (0.5732, 0.3643)},
    9: {"L_max": 128, "terpotong": {"latih": 0, "validasi": 0, "uji": 1}, "terpotong_id": [2741]},
    10: {"T": 764, "T_w": 76, "f1_di_atas": 0.3634, "ulang_seed42_toleransi": 0.0},
    11: {"kandidat": 50, "pelatihan": 100},
    12: {"evaluasi": 50, "angka_acak": 300},
    13: {"M": 100},
    14: {"pelatihan": 9, "determinisme_toleransi": 1e-4},
    15: {"n_uji": 656, "acuan": 0.3643},
    16: {"z1": 2.3940, "z2": 0.8416, "pengganda": 3.2356, "persentil": (0.8333, 99.1667),
         "verifikasi_toleransi": 1e-12},
}

# 6) Fungsi bantu
_STATUS = {"cek_mulai": False}
_ANGKA = (int, float, np.integer, np.floating)

def _angka(x):
    return isinstance(x, _ANGKA) and not isinstance(x, (bool, np.bool_))

def fmt_id(x, d=None):
    """Format angka Indonesia: 7204 -> '7.204'; 0.836 -> '0,8360'. Bukan angka -> str(x)."""
    if not _angka(x):
        return str(x)
    if d is None:
        d = 0 if isinstance(x, (int, np.integer)) else 4
    return f"{x:,.{d}f}".replace(",", "_").replace(".", ",").replace("_", ".")

def _tampil(v, d=None):
    if isinstance(v, np.ndarray):
        v = v.tolist()
    if isinstance(v, (list, tuple)) and v and all(_angka(x) for x in v):
        return " · ".join(fmt_id(x, d) for x in v)
    if isinstance(v, dict):
        return ", ".join(f"{k}: {_tampil(x, d)}" for k, x in v.items())
    return fmt_id(v, d)

def garis():
    print("─" * 46)

def judul(no, nama, masukan=None, keluaran=None):
    """Kepala blok ringkasan; masukan/keluaran boleh str atau list of str."""
    _STATUS["cek_mulai"] = False
    print(f"══════════ TAHAP {no} — {nama.upper()} ══════════")
    for label, isi in (("Masukan ", masukan), ("Keluaran", keluaran)):
        if isi:
            isi = [isi] if isinstance(isi, str) else list(isi)
            print(f"{label}: {isi[0]}")
            for x in isi[1:]:
                print(" " * 10 + x)
    garis()

def cek(nama, aktual, target, toleransi=0, cara="=", d=None):
    """Cetak ✅/⚠️ tanpa menghentikan program. cara: '=', '>', '>=', '<', '<=', 'memuat'."""
    if not _STATUS["cek_mulai"]:
        garis()
        print("Cek otomatis:")
        _STATUS["cek_mulai"] = True
    if _angka(aktual) and _angka(target):
        if d is None and toleransi > 0:
            d = max(0, round(-math.log10(2 * toleransi)))
        a, t = float(aktual), float(target)
        ok = {"=": abs(a - t) <= toleransi, ">": a > t, ">=": a >= t, "<": a < t, "<=": a <= t}[cara]
    elif cara == "memuat":
        ok = str(target) in str(aktual)
    else:
        try:
            ok = bool(aktual == target)
        except ValueError:                       # array numpy
            ok = bool(np.array_equal(aktual, target))
    tanda = "" if cara in ("=", "memuat") else cara + " "
    print(f"  {'✅' if ok else '⚠️'} {nama} = {_tampil(aktual, d)} (target {tanda}{_tampil(target, d)})")
    return bool(ok)

def selesai(t0):
    dt = time.perf_counter() - t0
    teks = (f"{fmt_id(dt, 1)} detik" if dt < 120 else
            f"{fmt_id(dt / 60, 1)} menit" if dt < 7200 else f"{fmt_id(dt / 3600, 2)} jam")
    print(f"Waktu: {teks}")

def rel(p):
    """Path relatif terhadap folder kerja, untuk ditampilkan."""
    try:
        return str(Path(p).relative_to(PATH["akar"]))
    except ValueError:
        return str(p)

def simpan_tabel(df, nama, index=False):
    """CSV untuk Bab IV (angka mentah) -> bab4/tabel/<nama>.csv"""
    p = PATH["tabel"] / f"{nama}.csv"
    df.to_csv(p, index=index)
    return p

def simpan_gambar(fig, nama):
    """PNG 300 dpi -> bab4/gambar/<nama>.png, lalu ditampilkan satu kali."""
    p = PATH["gambar"] / f"{nama}.png"
    fig.savefig(p, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return p

def _ke_json(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, (np.ndarray, Path)):
        return o.tolist() if isinstance(o, np.ndarray) else str(o)
    raise TypeError(f"tidak bisa disimpan ke JSON: {type(o)}")

def simpan_json(obj, path):
    """Tulis JSON secara atomik (berkas .tmp lalu diganti), aman bila Colab terputus."""
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=_ke_json)
    os.replace(tmp, path)
    return path

def baca_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def atur_seed(s):
    """Satu seed untuk semua sumber keacakan pelatihan (Bagian 3.4)."""
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    transformers.set_seed(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)     # Keputusan no. 10

# 7) GPU dan versi package -> log/versi_package.json
def _versi(pip):
    try:
        return _versi_pkg(pip)
    except Exception:
        return "tidak ditemukan"

_ada_gpu = torch.cuda.is_available()
_gpu = torch.cuda.get_device_name(0) if _ada_gpu else "tidak ada"
_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3 if _ada_gpu else 0.0
_sesi = {
    "waktu": datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "gpu": _gpu, "gpu_memori_gb": round(_mem, 2),
    "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version() if _ada_gpu else None,
    "package": {pip: _versi(pip) for pip in [*PAKET.values(), "tokenizers"]},
}
_berkas_log = PATH["log"] / "versi_package.json"
_pertama = baca_json(_berkas_log)["pertama"] if _berkas_log.exists() else _sesi
simpan_json({"pertama": _pertama, "terbaru": _sesi}, _berkas_log)
_berubah = [k for k in ("torch", "transformers", "tokenizers") if _pertama["package"].get(k) != _sesi["package"].get(k)]
if _pertama["gpu"] != _sesi["gpu"]:
    _berubah.append("gpu")

# 8) Ringkasan
_ada_data = PATH["xlsx"].exists()
judul(0, "Persiapan Lingkungan",
      masukan=f"{rel(PATH['xlsx'])} ({fmt_id(PATH['xlsx'].stat().st_size / 1024, 1)} KB)" if _ada_data
              else f"{rel(PATH['xlsx'])} (BELUM ADA)",
      keluaran=rel(_berkas_log))
print(f"Folder kerja : {AKAR}")
print(f"Python       : {_sesi['python']}")
print(f"GPU          : {_gpu} ({fmt_id(_mem, 1)} GB) · CUDA {_sesi['cuda']} · cuDNN {_sesi['cudnn']}")
if _kurang:
    print(f"Dipasang     : {', '.join(_kurang)}")
print()
print(f"{'Package':<14}Versi")
for _k, _v in _sesi["package"].items():
    print(f"{_k:<14}{_v}")
cek("berkas data ada", "ya" if _ada_data else "tidak", "ya")
cek("GPU", _gpu, "T4", cara="memuat")
cek("package tersedia", sum(_versi(p) != "tidak ditemukan" for p in PAKET.values()), len(PAKET))
cek("CUBLAS_WORKSPACE_CONFIG sebelum CUDA aktif",
    "tidak (restart runtime)" if (_cuda_sudah_aktif and _cublas_lama != ":4096:8") else "ya", "ya")
cek("mode deterministik PyTorch", "aktif" if torch.are_deterministic_algorithms_enabled() else "mati", "aktif")
cek("versi torch/transformers/tokenizers/GPU sama dengan sesi pertama",
    "sama" if not _berubah else "berubah: " + ", ".join(_berubah), "sama")
selesai(t0)

## Tahap 1 — Pengumpulan Data

**Package:** `pandas` (baca .xlsx, olah tabel), `openpyxl` (mesin baca .xlsx untuk pandas), `pyarrow` (mesin simpan .parquet untuk pandas), `time` (bawaan Python).

Membaca data mentah, membuang kolom indeks `Unnamed*`, lalu menyimpannya ke `hasil/tahap01_mentah.parquet` tanpa mengubah isi. `review_id` dibuat di Tahap 3 (Keputusan no. 6, Bagian 9 `CLAUDE.md`).

In [ ]:
# ══ Tahap 1 — Pengumpulan Data ══
# Package : pandas (baca .xlsx, olah tabel), openpyxl (mesin baca .xlsx untuk pandas),
#           pyarrow (mesin simpan .parquet untuk pandas), time (bawaan Python)
import time
import pandas as pd

t0 = time.perf_counter()
T = TARGET[1]
BULAN = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli",
         "Agustus", "September", "Oktober", "November", "Desember"]

def tgl_id(t):                                   # 2026-07-19 -> "19 Juli 2026"
    return f"{t.day} {BULAN[t.month - 1]} {t.year}"

# 1) Baca data mentah (.xlsx)
mentah = pd.read_excel(PATH["xlsx"], engine="openpyxl")

# 2) Buang kolom indeks bawaan Excel (Unnamed*); isi data tidak diubah, review_id dibuat di Tahap 3
buang = [c for c in mentah.columns if str(c).startswith("Unnamed")]
df = mentah.drop(columns=buang)

# 3) Simpan keluaran, lalu pastikan berkas yang dibaca ulang identik
keluar = PATH["hasil"] / "tahap01_mentah.parquet"
df.to_parquet(keluar, index=False)
identik = df.equals(pd.read_parquet(keluar))

# 4) Ringkasan + cek otomatis
tgl = pd.to_datetime(df["tanggal"])              # hanya untuk ringkasan; kolom asli tidak diubah
per_bulan = tgl.dt.strftime("%Y-%m").value_counts().sort_index()
versi = sorted(df["versi"].astype(str).unique())

judul(1, "Pengumpulan Data",
      masukan=f"{rel(PATH['xlsx'])} ({fmt_id(len(mentah))} baris, {mentah.shape[1]} kolom)",
      keluaran=f"{rel(keluar)} ({fmt_id(len(df))} baris, {df.shape[1]} kolom)")
print(f"Kolom dibuang  : {', '.join(map(str, buang)) if buang else '-'}")
print(f"Rentang tanggal: {tgl_id(tgl.min())} – {tgl_id(tgl.max())}")
print(f"Versi aplikasi : {', '.join(versi)}")
print()
print(f"{'Bulan':<16}{'Ulasan':>7}")
for k, v in per_bulan.items():
    print(f"{BULAN[int(k[5:]) - 1] + ' ' + k[:4]:<16}{fmt_id(v):>7}")
print(f"{'Total':<16}{fmt_id(int(per_bulan.sum())):>7}")

cek("jumlah baris", len(df), T["baris"])
cek("nama dan urutan kolom", "sesuai" if list(df.columns) == T["kolom"] else ", ".join(df.columns), "sesuai")
cek("rentang tanggal", f"{tgl.min().date()} s.d. {tgl.max().date()}", f"{T['tanggal'][0]} s.d. {T['tanggal'][1]}")
cek("nilai unik versi", ", ".join(versi), ", ".join(T["versi"]))
cek("ulasan per bulan", per_bulan.to_dict(), T["per_bulan"])
cek("parquet identik dengan isi .xlsx", "ya" if identik else "tidak", "ya")
selesai(t0)